In [2]:
import numpy as np
import pandas as pd
from datetime import date
import collections
import datetime
import os
import xarray as xr
import csv
import itertools
import glob

In [8]:
# Directory containing the subdirectories with CSV files
main_directory = '/Users/finnwimberly/Desktop/Lizz Research/CSV Outputs/Runoff/'

# Lists to store DataFrames
RF_dfs = []
RF_Aligned_dfs = []

# Iterate over all subdirectories in the main directory
for subdir in os.listdir(main_directory):
    subdir_path = os.path.join(main_directory, subdir)
    if os.path.isdir(subdir_path):
        # Iterate over all files in the subdirectory
        for filename in os.listdir(subdir_path):
            if filename.endswith('.csv'):
                # Full path to the file
                filepath = os.path.join(subdir_path, filename)
                
                # Read the CSV file
                df = pd.read_csv(filepath)
                
                # Add the trimmed filename as a new column
                if filename.startswith('runoff_AlignedMonthly_'):
                    trimmed_filename = filename[len('runoff_AlignedMonthly_'):-4]
                    RF_Aligned_dfs.append(df)
                elif filename.startswith('runoff_'):
                    trimmed_filename = filename[len('runoff_'):-4]
                    RF_dfs.append(df)
                
                df['source_file'] = trimmed_filename

# Concatenate all DataFrames into two separate DataFrames
all_rf_data = pd.concat(RF_dfs, ignore_index=True)
all_rf_aligned_data = pd.concat(RF_Aligned_dfs, ignore_index=True)

# Optionally, set the filename as the index for both DataFrames
all_rf_data.set_index('source_file', inplace=True)
all_rf_aligned_data.set_index('source_file', inplace=True)

# Rename the index to 'GCM_SSP_Basin'
all_rf_data = all_rf_data.rename_axis('GCM_SSP_Basin')
all_rf_aligned_data = all_rf_aligned_data.rename_axis('GCM_SSP_Basin')

# Rename the first column to 'Date' in both DataFrames
all_rf_data.columns = ['Date'] + list(all_rf_data.columns[1:])
all_rf_aligned_data.columns = ['Date'] + list(all_rf_aligned_data.columns[1:])

In [9]:
# Write the DataFrames to CSV files on the desktop
desktop_path = '/Users/finnwimberly/Desktop/'
all_rf_data.to_csv(os.path.join(desktop_path, 'all_rf_data.csv'))
all_rf_aligned_data.to_csv(os.path.join(desktop_path, 'all_rf_aligned_data.csv'))

In [10]:
all_rf_aligned_data

,Date,GloGEM,PyGEM,OGGM
GCM_SSP_Basin,,,,
MPI-ESM1-2-HR_ssp585_TARIM HE,1999-10,0.033061,0.000000,0.000000e+00
MPI-ESM1-2-HR_ssp585_TARIM HE,1999-11,0.000000,0.000000,0.000000e+00
MPI-ESM1-2-HR_ssp585_TARIM HE,1999-12,0.000000,0.000000,0.000000e+00
MPI-ESM1-2-HR_ssp585_TARIM HE,2000-01,0.000000,0.000000,2.757654e-17
MPI-ESM1-2-HR_ssp585_TARIM HE,2000-02,0.000000,0.000000,2.034425e-17
...,...,...,...,...
FGOALS-f3-L_ssp126_COLUMBIA,2100-08,1.145592,0.815447,0.000000e+00
FGOALS-f3-L_ssp126_COLUMBIA,2100-09,0.694784,0.504670,0.000000e+00
FGOALS-f3-L_ssp126_COLUMBIA,2100-10,0.000000,0.002133,0.000000e+00


Code for users to read in data:

In [11]:
# Path to the single CSV file
csv_file_path = '/Users/finnwimberly/Desktop/Lizz Research/CSV Outputs/Runoff/all_rf_data.csv'

# Read the CSV file
df = pd.read_csv(csv_file_path)

# Dictionary to store DataFrames
RF_dict = {}

# Iterate over the rows and populate the dictionary
for index, row in df.iterrows():
    # Extract GCM, SSP, and Basin from the source_file column
    gcm_ssp_basin = row['source_file'].split('_')
    gcm = gcm_ssp_basin[0]
    ssp = gcm_ssp_basin[1]
    basin = gcm_ssp_basin[2]
    
    # Initialize nested dictionary structure if not already done
    if gcm not in RF_dict:
        RF_dict[gcm] = {}
    if ssp not in RF_dict[gcm]:
        RF_dict[gcm][ssp] = {}
    if basin not in RF_dict[gcm][ssp]:
        RF_dict[gcm][ssp][basin] = []
    
    # Append the row to the corresponding DataFrame
    RF_dict[gcm][ssp][basin].append(row)

# Convert lists of rows to DataFrames
for gcm in RF_dict:
    for ssp in RF_dict[gcm]:
        for basin in RF_dict[gcm][ssp]:
            # Convert list of rows to DataFrame
            temp_df = pd.DataFrame(RF_dict[gcm][ssp][basin])
            # Drop the 'source_file' column
            temp_df.drop(columns=['source_file'], inplace=True)
            # Set the 'Date' column as the index
            temp_df.set_index('Date', inplace=True)
            # Store the DataFrame back in the dictionary
            RF_dict[gcm][ssp][basin] = temp_df

In [12]:
RF_dict['BCC-CSM2-MR']['ssp126']['RHONE']

,GloGEM,PyGEM,OGGM
Date,,,
2000-01,0.114951,0.000000e+00,0.000057
2000-02,0.000000,0.000000e+00,0.000414
2000-03,0.000000,1.250834e-07,0.000610
2000-04,0.000000,1.627167e-02,0.062100
2000-05,0.000000,4.611179e-02,0.167005
...,...,...,...
2100-08,0.192996,2.610112e-01,0.000000
2100-09,0.684146,6.469116e-02,0.000000
2100-10,0.787569,1.360359e-02,0.000000
